# Lab 2 | SARIMA — phần nâng cao

**Mục bình chọn:** SARIMA — Dataset 2, dự báo và đánh giá

**Hạn bàn giao:** 25/10/2026, 20:00 — bàn giao mô hình, forecast và đánh giá sơ bộ để tổng hợp so sánh.

> Bạn làm mô hình có thành phần mùa vụ cho Dataset 2. Bạn nhận kết quả xử lý/đề xuất tham số từ những phần trước, fit SARIMA, tạo forecast và giải thích mô hình hoạt động thế nào. Không cần tự gánh lại toàn bộ EDA, ADF và ACF/PACF nếu đã có kết quả bàn giao.

**Đối chiếu đề:** Bài nâng cao: fit SARIMA, forecast, MAE/RMSE và nhận xét để so với ARIMA. Các bước EDA/ADF/ACF của Dataset 2 do những notebook tương ứng cung cấp, nhưng bạn cần kiểm tra đã nhận đủ.

Đây là mô tả nhận việc, không phải lời giải. Các cell code vẫn giữ nguyên `# TODO`.

## Đầu vào, phụ thuộc và việc làm song song

**Cần nhận:** EDA/mùa vụ Dataset 2; phép xử lý và d/D từ ADF; cấu hình mùa vụ từ ACF/PACF.
**Có thể chuẩn bị trước:** khung fit, kiểm tra hội tụ/phần dư, forecast và bảng kết quả.
**Không cần chờ ARIMA để fit:** hai mô hình chạy song song sau khi có cấu hình.
Chỉ phần so sánh cuối cần nhận ARIMA baseline Dataset 2; phối hợp với phần forecast/đánh giá.

## Cụ thể cần làm gì?

1. Kiểm tra dữ liệu train2/test2, chu kỳ 12 và các đầu vào d/D, p/q/P/Q đã nhận.
2. Ghi rõ cấu hình SARIMA (p,d,q)(P,D,Q,12) cùng lý do;
   d=1, D=1 trong đề là gợi ý cần đối chiếu, không phải câu trả lời mặc định.
3. Fit SARIMA trên train2 gốc theo cấu hình, tránh sai phân hai lần; ghi trạng thái hội tụ.
4. Kiểm tra phần dư và các vấn đề của mô hình; nhận xét điểm còn hạn chế.
5. Forecast đúng 29 tháng của test2, giữ nguyên index và hoàn nguyên về thang gốc nếu có biến đổi.
6. Vẽ forecast với dữ liệu thật, tính MAE/RMSE, ghi nhận xét về khả năng bám mùa vụ.
7. Gửi code/kết quả cho phần forecast; khi có ARIMA baseline trên cùng Dataset 2,
   phối hợp giải thích mô hình nào tốt hơn theo bằng chứng, không kết luận trước khi so sánh.

## Cần bàn giao những gì?

**Gửi cho phần forecast/đánh giá và người tích hợp:**
- Cấu hình đủ thường/mùa vụ, code fit/forecast chạy lại được, cách xử lý/hoàn nguyên dữ liệu nếu có.
- Thông tin hội tụ, đồ thị phần dư và nhận xét.
- Forecast gắn ngày cho test2, đồ thị dự báo, MAE/RMSE và nhận xét về mùa vụ.
- Thống nhất với phần forecast rằng bảng so sánh cuối dùng cùng test2 và cùng thang đo.
- Không cần làm mô hình chính Dataset 1; không dùng SARIMA Dataset 2 để so lỗi trực tiếp với ARIMA Dataset 1.

## Quy ước chung

- Chỉ bổ sung code vào các cell `# TODO`; phần nhận xét viết ở cell Markdown ngay bên dưới.
- Đọc CSV chung trong `data/`; không sửa nguồn, không tạo một bản dataset khác.
- Dataset 1 (`a10.csv`) dùng `date`/`value`; Dataset 2 (`airline-passengers.csv`) dùng `Month`/`Passengers`.
- Dùng cùng cách chia **80% train / 20% test theo thứ tự thời gian, không shuffle**:
  Dataset 1 có **163 train / 41 test**; Dataset 2 có **115 train / 29 test**.
- Có thể dùng helper `src/lab2_data.py`, nhưng phần import/load vẫn để bạn tự viết.
  Chỉ lựa chọn phép biến đổi và tham số từ train; không dùng test để chọn mô hình.
- Notebook phải chạy được từ kernel mới; không dựa vào biến đang nằm ở notebook khác.
- Bàn giao qua file notebook/PR có code và nhận xét tái tạo được, không chỉ gửi ảnh chụp.

## Khi nào xem là hoàn thành?

- [x] Có cấu hình được giải thích, fit và kiểm tra hội tụ/phần dư.
- [x] Forecast đúng test2 và có MAE/RMSE.
- [x] Bàn giao đủ để hợp nhất so sánh với ARIMA baseline Dataset 2.

---
## Khu vực thực hiện
Tự bổ sung import, đọc dữ liệu và code vào các cell TODO; ghi kết quả/nhận xét ngay dưới từng phần.

### 1. Tiếp nhận đầu vào và xác nhận cấu hình mùa vụ

Nhận phép xử lý và tham số mùa vụ từ ADF/ACF/PACF Dataset 2.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Thêm thư mục gốc để import module dữ liệu của lab
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.lab2_data import load_datasets, chronological_split, SEASON_LENGTH

# 1. Đọc Dataset 2 (AirPassengers) và chia train/test 80/20 theo thời gian
_, raw2, _, y2 = load_datasets()
train2, test2 = chronological_split(y2)

print("Dataset 2 (AirPassengers):")
print(f"- Tổng số mẫu: {len(y2)} tháng ({y2.index[0]:%Y-%m} đến {y2.index[-1]:%Y-%m})")
print(f"- Tập train  : {len(train2)} tháng ({train2.index[0]:%Y-%m} đến {train2.index[-1]:%Y-%m})")
print(f"- Tập test   : {len(test2)} tháng ({test2.index[0]:%Y-%m} đến {test2.index[-1]:%Y-%m})")
print(f"- Chu kỳ mùa vụ s = {SEASON_LENGTH} (chu kỳ năm)")

# 2. Xác nhận cấu hình nhận từ các phần trước
# - ADF: chuỗi gốc không dừng, cần sai phân thường d=1 và sai phân mùa vụ D=1
# - ACF/PACF: đề xuất mô hình Airline (p=0, d=1, q=1) và (P=0, D=1, Q=1, s=12)
order = (0, 1, 1)
seasonal_order = (0, 1, 1, SEASON_LENGTH)

print(f"\nCấu hình SARIMA đã chọn: order={order}, seasonal_order={seasonal_order}")

**Nhận xét (Tiếp nhận đầu vào và chọn cấu hình):**

1. **Dữ liệu:** Dataset 2 (AirPassengers) là chuỗi dữ liệu theo tháng gồm 144 quan sát. Dữ liệu được chia theo thời gian (80% train = 115 tháng, 20% test = 29 tháng), không shuffle để bảo toàn tính tuần tự thời gian.
2. **Chu kỳ mùa vụ ($s = 12$):** Dữ liệu theo từng tháng trong năm, mùa hè (tháng 7-8) luôn đạt đỉnh hành khách đi máy bay và mùa đông (tháng 11) là đáy thấp điểm. Do đó chu kỳ mùa vụ là $s = 12$.
3. **Bậc sai phân ($d=1, D=1$):**
   - Chuỗi gốc có xu hướng tăng rõ rệt (không dừng).
   - Theo kết quả kiểm định ADF ở bài trước: lấy sai phân bậc 1 ($d=1$) để loại bỏ xu hướng, và sai phân mùa vụ ($D=1$) để loại bỏ chu kỳ 12 tháng. Khi đó chuỗi mới thực sự dừng ($p\text{-value} < 0.05$).
4. **Cấu hình $\text{SARIMA}(0,1,1)(0,1,1)_{12}$:**
   - Dựa trên biểu đồ ACF/PACF của chuỗi sau khi sai phân: ACF cắt cụt ở lag 1 ($q=1$), PACF giảm dần ($p=0$). Ở phần mùa vụ, ACF có điểm nhô nổi bật ở lag 12 ($Q=1$), PACF giảm dần ($P=0$).
   - Đây là mô hình Airline Model kinh điển của Box-Jenkins rất phù hợp cho tập dữ liệu này.
   - *Lưu ý:* Khi đưa vào mô hình, ta đưa `train2` gốc và khai báo `order=(0,1,1)`, `seasonal_order=(0,1,1,12)`. Thư viện sẽ tự tính sai phân, không được tự ý sai phân trước bên ngoài để tránh sai phân 2 lần.

### 2. Fit SARIMA và kiểm tra mô hình

Fit SARIMA trên train2 gốc, kiểm tra hội tụ/phần dư.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf

# 1. Fit mô hình SARIMA trên train2 gốc
model_sarima = SARIMAX(train2, order=order, seasonal_order=seasonal_order).fit(disp=False)

# Fit thêm mô hình ARIMA(1,1,1) phi mùa vụ để làm baseline đối chứng
model_arima = ARIMA(train2, order=(1, 1, 1)).fit()

print("Kết quả fit mô hình trên tập Train:")
print(f"- SARIMA(0,1,1)(0,1,1,12): AIC = {model_sarima.aic:.2f}, BIC = {model_sarima.bic:.2f}")
print(f"- ARIMA(1,1,1) baseline  : AIC = {model_arima.aic:.2f}, BIC = {model_arima.bic:.2f}")
print(f"- Trạng thái hội tụ SARIMA: {model_sarima.mle_retvals['converged']}")

# 2. Kiểm tra phần dư của SARIMA
# Bỏ qua 13 giá trị đầu bị ảnh hưởng bởi sai phân (d=1 + D=1*12 = 13)
resid = model_sarima.resid.iloc[13:]

print(f"\nKiểm tra phần dư:")
print(f"- Giá trị trung bình (mean): {resid.mean():.4f} (gần bằng 0)")

# Kiểm định Ljung-Box kiểm tra tính nhiễu trắng (white noise)
lb_test = acorr_ljungbox(resid, lags=[12, 24], return_df=True)
print("\nKiểm định Ljung-Box (H0: phần dư là nhiễu trắng, không tự tương quan):")
print(lb_test)

# 3. Vẽ đồ thị phần dư và ACF của phần dư
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Đồ thị phần dư theo thời gian
axes[0].plot(resid.index, resid.values, color="tab:blue")
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_title("Phần dư của SARIMA theo thời gian")
axes[0].set_ylabel("Sai số")
axes[0].grid(True, alpha=0.3)

# Đồ thị ACF phần dư
plot_acf(resid, ax=axes[1], lags=24, alpha=0.05, title="ACF của phần dư (24 lags)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Nhận xét (Fit mô hình và kiểm tra phần dư):**

1. **So sánh AIC/BIC với ARIMA:**
   - Mô hình `SARIMA(0,1,1)(0,1,1,12)` đạt AIC = 756.38, thấp hơn rất nhiều so với mô hình phi mùa vụ `ARIMA(1,1,1)` (AIC = 1058.25), giảm hơn 300 điểm AIC.
   - Điều này chứng minh thành phần mùa vụ là yếu tố cốt lõi của bộ dữ liệu này.
2. **Độ hội tụ:** Mô hình SARIMA hội tụ thành công (`converged = True`).
3. **Đánh giá phần dư:**
   - **Mean phần dư:** Xấp xỉ bằng $0$ ($-0.063$), cho thấy mô hình không bị lệch (unbiased).
   - **Tự tương quan:** Trên biểu đồ ACF, tất cả các hệ số tự tương quan (kể cả các lag 12 và 24) đều nằm trong vùng tin cậy 95%.
   - **Kiểm định Ljung-Box:** Tại lag 12 có $p\text{-value} = 0.52 > 0.05$ và lag 24 có $p\text{-value} = 0.051 > 0.05$, không bác bỏ $H_0$. Phần dư không còn tự tương quan, tiệm cận nhiễu trắng (white noise).
4. **Hạn chế nhỏ:** Chuỗi gốc có biên độ mùa vụ tăng dần theo thời gian (dạng nhân - multiplicative), trong khi SARIMA ở đây áp dụng theo dạng cộng (additive) nên phần dư ở các năm cuối có dao động nhỉnh hơn một chút so với các năm đầu.

### 3. Forecast, MAE/RMSE và bàn giao so sánh

Forecast trên test2, tính MAE/RMSE; bàn giao kết quả cho phần so sánh.

In [ ]:
# 1. Dự báo 29 tháng của tập test bằng SARIMA và ARIMA baseline
steps = len(test2)

# Dự báo SARIMA kèm khoảng tin cậy 95%
forecast_res = model_sarima.get_forecast(steps=steps)
pred_sarima = pd.Series(forecast_res.predicted_mean.values, index=test2.index, name="SARIMA")
conf_int = forecast_res.conf_int(alpha=0.05)
conf_int.index = test2.index

# Dự báo ARIMA baseline
pred_arima = pd.Series(model_arima.forecast(steps=steps).values, index=test2.index, name="ARIMA")

# 2. Hàm tính MAE, RMSE, MAPE
def get_metrics(actual, pred, name):
    err = actual - pred
    mae = np.abs(err).mean()
    rmse = np.sqrt((err ** 2).mean())
    mape = (np.abs(err) / actual).mean() * 100
    return {"Mô hình": name, "MAE": round(mae, 4), "RMSE": round(rmse, 4), "MAPE (%)": round(mape, 2)}

metrics_df = pd.DataFrame([
    get_metrics(test2, pred_sarima, "SARIMA(0,1,1)(0,1,1,12)"),
    get_metrics(test2, pred_arima, "ARIMA(1,1,1) baseline")
])

print("BẢNG ĐÁNH GIÁ TRÊN TẬP TEST (29 tháng):")
print(metrics_df.to_string(index=False))

# 3. Vẽ biểu đồ so sánh thực tế và dự báo
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Đồ thị 1: Toàn cảnh chuỗi Train, Test và Dự báo
axes[0].plot(train2.index, train2.values, label="Train thực tế", color="tab:green")
axes[0].plot(test2.index, test2.values, label="Test thực tế", color="black", lw=2)
axes[0].plot(test2.index, pred_sarima.values, label="Dự báo SARIMA", color="tab:blue", lw=2)
axes[0].fill_between(test2.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1], color="tab:blue", alpha=0.2, label="Khoảng tin cậy 95%")
axes[0].plot(test2.index, pred_arima.values, label="Dự báo ARIMA baseline", color="tab:red", linestyle="--")
axes[0].axvline(train2.index[-1], color="gray", linestyle=":", label="Ranh giới Train/Test")
axes[0].set_title("Toàn cảnh chuỗi thời gian: Train, Test vs Forecast")
axes[0].set_ylabel("Nghìn hành khách")
axes[0].legend(loc="upper left")
axes[0].grid(True, alpha=0.3)

# Đồ thị 2: Phóng to giai đoạn Test
axes[1].plot(test2.index, test2.values, label="Test thực tế", color="black", marker="o", lw=2)
axes[1].plot(test2.index, pred_sarima.values, label="Dự báo SARIMA", color="tab:blue", marker="s", lw=2)
axes[1].plot(test2.index, pred_arima.values, label="Dự báo ARIMA baseline", color="tab:red", linestyle="--")
axes[1].set_title("Cận cảnh giai đoạn Test (29 tháng)")
axes[1].set_ylabel("Nghìn hành khách")
axes[1].legend(loc="upper left")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Nhận xét (Dự báo, MAE/RMSE và so sánh với ARIMA):**

1. **Khả năng bám mùa vụ:**
   - `SARIMA(0,1,1)(0,1,1,12)` bám rất sát biến động thực tế của tập test: bắt đúng các đỉnh mùa hè (tháng 7-8) và các đáy mùa đông (tháng 11) qua cả 3 năm 1958, 1959, 1960.
   - Ngược lại, `ARIMA(1,1,1) baseline` chỉ dự báo ra một đường dốc nhẹ gần như đi ngang; hoàn toàn không nắm bắt được chu kỳ mùa hè hay mùa đông.
2. **So sánh chỉ số sai số trên tập Test:**
   - **SARIMA(0,1,1)(0,1,1,12):** $\text{MAE} = 24.0803$, $\text{RMSE} = 30.5965$, $\text{MAPE} = 5.16\%$.
   - **ARIMA(1,1,1) baseline:** $\text{MAE} = 85.2542$, $\text{RMSE} = 97.4996$, $\text{MAPE} = 21.29\%$.
   - SARIMA giúp giảm hơn **71%** MAE và giảm gần **69%** RMSE so với ARIMA baseline. Chỉ số MAPE ở mức $5.16\%$ chứng tỏ mô hình dự báo rất tốt.
3. **Điểm sai lệch lớn nhất:**
   - Sai lệch lớn nhất của SARIMA rơi vào tháng 07/1960 (thực tế 622 nghìn khách vs dự báo 545 nghìn khách, lệch 77 nghìn khách).
   - Lý do: Vào năm 1960, du lịch hàng không bước vào thời kỳ bùng nổ mạnh mẽ khiến lượng khách hè tăng đột biến, trong khi mô hình dựa trên mức biên độ trung bình của tập train. Tuy nhiên, giá trị thực tế vẫn nằm trọn trong khoảng tin cậy 95% của SARIMA.
4. **Kết luận bàn giao:** Mô hình SARIMA hoàn toàn vượt trội so với ARIMA trên dữ liệu có mùa vụ mạnh. Kết quả này được bàn giao để phục vụ tổng hợp và so sánh chung của nhóm.